# Style Similarity Search V5 · DINOv2 × Qwen3-VL Hybrid

Merges the best of V4 (hybrid visual + conceptual clustering, text search, curator labels) and the DINOv2 pipeline (style-focused backbone, tuned clustering, fully local — no Ollama).

| Signal | Model | Weight | What it captures |
|---|---|---|---|
| Visual | `DINOv2-large` (1024-dim) | 40% | Texture, grain, tonal range, composition |
| Conceptual | `Qwen3-VL-2B-Instruct` → `BGE-large-en-v1.5` (1024-dim) | 60% | Aesthetic mood, style vocabulary, vibe |

**Download models if not present:**
```bash
huggingface-cli download facebook/dinov2-large --local-dir ./models/dinov2-large
huggingface-cli download BAAI/bge-large-en-v1.5 --local-dir ./models/bge-large-en-v1.5
```

**Hardware:** Apple M4 Pro · 24 GB Unified Memory

In [7]:
import os, gc, json, re, warnings, hashlib
from pathlib import Path
from collections import defaultdict

import torch
import duckdb
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from transformers import (
    AutoImageProcessor, AutoModel, AutoTokenizer,
    Qwen3VLForConditionalGeneration, AutoProcessor
)

# --- Config ---
LIMIT         = 'All'
DATA_DIR      = Path('./data/raw')
DB_PATH       = Path('./outputs/photos.duckdb')
MODELS_DIR    = Path('./models')

VISUAL_WEIGHT = 0.4
TEXT_WEIGHT   = 0.6

# Caption model: '2b' fits entirely on MPS (~4 GB).
#                '8b' uses device_map=auto (MPS + CPU) for the 16 GB model.
CAPTION_MODEL = '2b'

# BGE path: use local copy if downloaded, otherwise pull from Hub (cached in ~/.cache/huggingface)
_bge_local = MODELS_DIR / 'bge-large-en-v1.5'
BGE_PATH   = str(_bge_local) if _bge_local.exists() else 'BAAI/bge-large-en-v1.5'

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'[✓] Device: {DEVICE}')
print(f'[✓] Caption model: Qwen3-VL-{CAPTION_MODEL.upper()}-Instruct')
print(f'[✓] BGE path: {BGE_PATH}')

[✓] Device: mps
[✓] Caption model: Qwen3-VL-2B-Instruct
[✓] BGE path: models/bge-large-en-v1.5


In [ ]:
def get_file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read(65536))
    return h.hexdigest()[:16]

def safe_to_rgb(path):
    img = Image.open(path)
    if img.mode in ('RGBA', 'LA', 'PA'):
        bg = Image.new('RGB', img.size, (255, 255, 255))
        if img.mode == 'PA':
            img = img.convert('RGBA')
        bg.paste(img, mask=img.split()[-1])
        return bg
    return img.convert('RGB')

def discover_images(directory):
    valid_exts = {'.jpg', '.jpeg', '.png', '.tiff', '.webp'}
    return sorted(f for f in directory.rglob('*') if f.suffix.lower() in valid_exts and f.is_file())

def extract_json(text):
    # Strip Qwen3 thinking blocks if present
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    m = re.search(r'\{[\s\S]*\}', text)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return None

def get_cc(p):
    """Return curator_commentary as a normalized dict regardless of model output shape.
    The 2B model flattens the schema: curator_commentary → style string,
    mood_profile/narrative_caption/keywords become top-level keys."""
    if not isinstance(p, dict):
        return {'style': '', 'mood_profile': '', 'narrative_caption': '', 'keywords': []}
    cc = p.get('curator_commentary', {})
    if isinstance(cc, dict):
        return cc
    return {
        'style':             str(cc) if cc else '',
        'mood_profile':      p.get('mood_profile', ''),
        'narrative_caption': p.get('narrative_caption', ''),
        'keywords':          p.get('keywords', []) if isinstance(p.get('keywords'), list) else [],
    }

def profile_to_text(p):
    tech = p.get('technical', {}) if isinstance(p, dict) else {}
    cc   = get_cc(p)
    parts = [
        tech.get('lighting', ''),
        tech.get('grain_texture', ''),
        tech.get('tonal_range', ''),
        tech.get('composition', ''),
        cc.get('style', ''),
        cc.get('mood_profile', ''),
        cc.get('narrative_caption', ''),
        ' '.join(cc.get('keywords', [])),
    ]
    return '. '.join(x for x in parts if x)

images_paths = discover_images(DATA_DIR)
if isinstance(LIMIT, int):
    images_paths = images_paths[:LIMIT]
print(f'[✓] {len(images_paths)} images found.')

In [3]:
print('[i] Stage 1: DINOv2-large visual embeddings...')
con = duckdb.connect(str(DB_PATH))
con.execute('''
    CREATE TABLE IF NOT EXISTS v5_visual (
        photo_id   VARCHAR PRIMARY KEY,
        image_path VARCHAR NOT NULL,
        dinov2_emb FLOAT[] NOT NULL,
        created_at TIMESTAMP DEFAULT now()
    )
''')

ip = AutoImageProcessor.from_pretrained(str(MODELS_DIR / 'dinov2-large'))
m  = AutoModel.from_pretrained(str(MODELS_DIR / 'dinov2-large'), device_map='mps')
m.eval()

dinov2_embs, valid_paths = [], []
for path in tqdm(images_paths, desc='DINOv2'):
    pid = get_file_hash(path)
    row = con.execute('SELECT dinov2_emb FROM v5_visual WHERE photo_id = ?', [pid]).fetchone()
    if row:
        dinov2_embs.append(np.array(row[0]))
        valid_paths.append(path)
        continue
    try:
        img    = safe_to_rgb(path)
        inputs = ip(images=img, return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            out = m(**inputs)
        emb = out.last_hidden_state[:, 0, :].detach().cpu().float().numpy().flatten()
        emb /= (np.linalg.norm(emb) + 1e-8)
        con.execute('INSERT INTO v5_visual VALUES (?, ?, ?, now())',
                    [pid, str(path.resolve()), emb.tolist()])
        dinov2_embs.append(emb)
        valid_paths.append(path)
    except Exception as e:
        print(f'[!] {path.name}: {e}')

con.close()
del m, ip
torch.mps.empty_cache()
gc.collect()
dinov2_embs = np.array(dinov2_embs)
print(f'[✓] {len(dinov2_embs)} visual embeddings ready.')

[i] Stage 1: DINOv2-large visual embeddings...


DINOv2: 100%|██████████| 100/100 [00:00<00:00, 5287.36it/s]


[✓] 100 visual embeddings ready.


In [4]:
AESTHETIC_PROMPT = (
    'Analyze this photograph. Respond with a JSON object only — no explanation, no markdown.\n'
    'Required keys:\n'
    '  technical: lighting, grain_texture, tonal_range, composition\n'
    '  curator_commentary: style (one phrase), mood_profile, narrative_caption, keywords (list of 5)\n'
    'Return only the raw JSON object.'
)

if CAPTION_MODEL == '2b':
    caption_model_path = str(MODELS_DIR / 'Qwen3-VL-2B-Instruct')
    caption_device_map = 'mps'
elif CAPTION_MODEL == '8b':
    caption_model_path = str(MODELS_DIR / 'Qwen3-VL-8B-Instruct')
    caption_device_map = 'auto'
else:
    raise ValueError(f"Unknown CAPTION_MODEL: '{CAPTION_MODEL}'. Choose '2b' or '8b'.")

print(f'[i] Stage 2: {caption_model_path}...')
con = duckdb.connect(str(DB_PATH))
con.execute('''
    CREATE TABLE IF NOT EXISTS v5_aesthetic (
        photo_id       VARCHAR PRIMARY KEY,
        image_path     VARCHAR NOT NULL,
        aesthetic_json VARCHAR NOT NULL,
        text_emb       FLOAT[],
        created_at     TIMESTAMP DEFAULT now()
    )
''')

proc = AutoProcessor.from_pretrained(caption_model_path)
proc.tokenizer.padding_side = 'left'
# Set directly on the image_processor — passing max_pixels as a kwarg is silently ignored
# in newer transformers. 1280*28*28 ≈ 1M pixels, enough for aesthetic analysis.
proc.image_processor.max_pixels = 1280 * 28 * 28
print(f'[i] image_processor.max_pixels = {proc.image_processor.max_pixels:,}')

qwen = Qwen3VLForConditionalGeneration.from_pretrained(
    caption_model_path,
    torch_dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map=caption_device_map
)
qwen.eval()

FALLBACK = {
    'technical': {'lighting': '', 'grain_texture': '', 'tonal_range': '', 'composition': ''},
    'curator_commentary': {'style': '', 'mood_profile': '', 'narrative_caption': '', 'keywords': []}
}

# Hard resize before processor as a belt-and-suspenders guard against large images
VL_MAX_SIDE = 1024
def resize_for_vl(img):
    w, h = img.size
    if max(w, h) > VL_MAX_SIDE:
        scale = VL_MAX_SIDE / max(w, h)
        return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

for path in tqdm(valid_paths, desc=f'Qwen3-VL-{CAPTION_MODEL.upper()}'):
    pid = get_file_hash(path)
    if con.execute('SELECT 1 FROM v5_aesthetic WHERE photo_id = ?', [pid]).fetchone():
        continue
    out = inputs = None
    try:
        img      = resize_for_vl(safe_to_rgb(path))
        messages = [{'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': AESTHETIC_PROMPT}
        ]}]
        text   = proc.apply_chat_template(messages, tokenize=False,
                                          add_generation_prompt=True, enable_thinking=False)
        inputs = proc(text=[text], images=[img], return_tensors='pt', padding=True)
        inputs = {k: v.to(qwen.device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = qwen.generate(**inputs, max_new_tokens=220, do_sample=False,
                                pad_token_id=proc.tokenizer.eos_token_id)
        decoded = proc.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        profile = extract_json(decoded) or FALLBACK
        con.execute(
            'INSERT INTO v5_aesthetic (photo_id, image_path, aesthetic_json) VALUES (?, ?, ?)',
            [pid, str(path.resolve()), json.dumps(profile)]
        )
    except Exception as e:
        print(f'[!] {path.name}: {e}')
    finally:
        del out, inputs
        torch.mps.empty_cache()
        gc.collect()

con.close()
del qwen, proc
torch.mps.empty_cache()
gc.collect()
print('[✓] Aesthetic profiles ready.')

[i] Stage 2: models/Qwen3-VL-2B-Instruct...
[i] image_processor.max_pixels = 1,003,520


Qwen3-VL-2B:   0%|          | 0/100 [00:00<?, ?it/s]Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Qwen3-VL-2B: 100%|██████████| 100/100 [11:21<00:00,  6.81s/it]

[✓] Aesthetic profiles ready.


In [10]:
print('[i] Stage 3: BGE-large-en-v1.5 text embeddings...')
con     = duckdb.connect(str(DB_PATH))
pending = con.execute(
    'SELECT photo_id, aesthetic_json FROM v5_aesthetic WHERE text_emb IS NULL'
).fetchall()
print(f'[i] {len(pending)} profiles to embed.')

if pending:
    bge_tok = AutoTokenizer.from_pretrained(BGE_PATH)
    bge_m   = AutoModel.from_pretrained(BGE_PATH, device_map='mps')
    bge_m.eval()

    for pid, aj in tqdm(pending, desc='BGE'):
        try:
            profile = json.loads(aj)
            text    = profile_to_text(profile)
            inputs  = bge_tok(text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
            with torch.inference_mode():
                out = bge_m(**inputs)
            emb = out.last_hidden_state[:, 0, :].detach().cpu().float().numpy().flatten()
            emb /= (np.linalg.norm(emb) + 1e-8)
            con.execute('UPDATE v5_aesthetic SET text_emb = ? WHERE photo_id = ?',
                        [emb.tolist(), pid])
        except Exception as e:
            print(f'[!] {pid}: {e}')

    del bge_m, bge_tok
    torch.mps.empty_cache()
    gc.collect()

con.close()
print('[✓] Text embeddings ready.')

[i] Stage 3: BGE-large-en-v1.5 text embeddings...
[i] 100 profiles to embed.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1772.62it/s]
BertModel LOAD REPORT from: models/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BGE: 100%|██████████| 100/100 [00:00<00:00, 225621.52it/s]


[!] 8bea1ec325193ce8: 'str' object has no attribute 'get'
[!] 62639171a863c50d: 'str' object has no attribute 'get'
[!] d5d9da43c7bfa354: 'str' object has no attribute 'get'
[!] 9e133ecc49f7bb9a: 'str' object has no attribute 'get'
[!] ee30ecbb50688ee0: 'str' object has no attribute 'get'
[!] 0ceef19ec1cca169: 'str' object has no attribute 'get'
[!] fcbf2126d8c71553: 'str' object has no attribute 'get'
[!] 23fa869043fed7fc: 'str' object has no attribute 'get'
[!] 47a5fe6c1f406a7e: 'str' object has no attribute 'get'
[!] 71af4a26fd59ac5f: 'str' object has no attribute 'get'
[!] 5f6fe7622bbc6b3f: 'str' object has no attribute 'get'
[!] 03bd73c0ed550598: 'str' object has no attribute 'get'
[!] 733fc3d73b9afa46: 'str' object has no attribute 'get'
[!] 73e75e81e290b93a: 'str' object has no attribute 'get'
[!] 938765c43b914ad3: 'str' object has no attribute 'get'
[!] 144cb51449952fe5: 'str' object has no attribute 'get'
[!] b5006b506c397e5a: 'str' object has no attribute 'get'
[!] 29a4de085b

In [11]:
print('[i] Hybrid clustering: UMAP → HDBSCAN...')
import umap
from sklearn.cluster import HDBSCAN

con       = duckdb.connect(str(DB_PATH))
vis_rows  = con.execute('SELECT photo_id, image_path, dinov2_emb FROM v5_visual').fetchall()
text_rows = con.execute('SELECT photo_id, text_emb FROM v5_aesthetic WHERE text_emb IS NOT NULL').fetchall()
meta_rows = con.execute('SELECT photo_id, aesthetic_json FROM v5_aesthetic').fetchall()
con.close()

vis_by_id  = {r[0]: (r[1], np.array(r[2])) for r in vis_rows}
text_by_id = {r[0]: np.array(r[1]) for r in text_rows}
meta_by_id = {r[0]: r[1] for r in meta_rows}

shared_ids  = [pid for pid in vis_by_id if pid in text_by_id]
paths_arr   = [Path(vis_by_id[pid][0]) for pid in shared_ids]
vis_matrix  = np.array([vis_by_id[pid][1] for pid in shared_ids])
text_matrix = np.array([text_by_id[pid]   for pid in shared_ids])

combined = np.hstack([vis_matrix * VISUAL_WEIGHT, text_matrix * TEXT_WEIGHT])
print(f'[i] Combined matrix: {combined.shape}')

n       = len(combined)
n_neigh = min(8, n - 1)
n_comp  = min(15, n - 2)
init    = 'spectral' if n > 40 else 'random'

reducer = umap.UMAP(n_components=n_comp, n_neighbors=n_neigh, min_dist=0.0,
                    metric='cosine', init=init, random_state=42)
reduced = reducer.fit_transform(combined)

vis_reducer = umap.UMAP(n_components=2, n_neighbors=n_neigh, min_dist=0.0,
                        metric='cosine', random_state=42)
vis_2d = vis_reducer.fit_transform(combined)

m_size    = max(3, n // 15)
clusterer = HDBSCAN(min_cluster_size=m_size, min_samples=1,
                    cluster_selection_method='leaf', metric='euclidean')
labels    = clusterer.fit_predict(reduced)

clusters = defaultdict(list)
for path, label in zip(paths_arr, labels):
    clusters[int(label)].append(path)

n_real  = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)
print(f'[✓] {n_real} clusters, {n_noise} noise points.')

[i] Hybrid clustering: UMAP → HDBSCAN...
[i] Combined matrix: (0,)


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
print('[i] Visualizing...')
COLS = 6

# UMAP scatter
plt.figure(figsize=(12, 8))
sc = plt.scatter(vis_2d[:, 0], vis_2d[:, 1], c=labels, cmap='Spectral',
                 s=50, alpha=0.7, edgecolors='w', lw=0.5)
plt.colorbar(sc, label='Cluster')
plt.title('V5 Hybrid Embedding Space (DINOv2 × Qwen3-VL)', fontsize=14)
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()

# Full cluster galleries — all members in a grid, V4 style
cluster_results = []

for cid in sorted(clusters.keys()):
    member_paths = clusters[cid]
    n            = len(member_paths)
    rep_path     = member_paths[0]
    rep_pid      = get_file_hash(rep_path)
    rep_profile  = json.loads(meta_by_id.get(rep_pid, '{}'))
    rep_cc       = get_cc(rep_profile)

    cluster_results.append({
        'cluster_id':          cid,
        'representative_path': str(rep_path),
        'member_count':        n,
        'member_paths':        [str(p) for p in member_paths],
        'style':               rep_cc.get('style', ''),
        'mood':                rep_cc.get('mood_profile', ''),
        'caption':             rep_cc.get('narrative_caption', ''),
        'keywords':            rep_cc.get('keywords', [])
    })

    name        = 'Outliers (Unique Aesthetic Profiles)' if cid == -1 else f'Style Group {cid}'
    rows_needed = (n // COLS) + (1 if n % COLS else 0)

    fig, axes = plt.subplots(rows_needed, COLS, figsize=(18, 3.5 * rows_needed))
    axes_flat = np.array(axes).flatten()

    for i, p in enumerate(member_paths):
        pid    = get_file_hash(p)
        prof   = json.loads(meta_by_id.get(pid, '{}'))
        cc_i   = get_cc(prof)
        title  = Path(p).name + '\n' + cc_i.get('style', '') + '\n' + cc_i.get('mood_profile', '')
        axes_flat[i].imshow(safe_to_rgb(p))
        axes_flat[i].set_title(title, fontsize=6)
        axes_flat[i].axis('off')

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis('off')

    suptitle = f'{name} ({n} photos)'
    if rep_cc.get('style'):
        suptitle += '  ·  ' + rep_cc['style']
    plt.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
print('[i] Saving cluster results to DuckDB...')
con = duckdb.connect(str(DB_PATH))

con.execute('DROP TABLE IF EXISTS v5_clusters')
con.execute('''
    CREATE TABLE v5_clusters (
        cluster_id  INTEGER  NOT NULL,
        photo_id    VARCHAR  NOT NULL,
        image_path  VARCHAR  NOT NULL,
        is_rep      BOOLEAN  NOT NULL,
        style       VARCHAR,
        mood        VARCHAR,
        caption     VARCHAR,
        keywords    VARCHAR[],
        created_at  TIMESTAMP DEFAULT now()
    )
''')

for res in cluster_results:
    cid     = res['cluster_id']
    style   = res['style']
    mood    = res['mood']
    caption = res['caption']
    kws     = res['keywords']
    for img_path in res['member_paths']:
        pid    = get_file_hash(Path(img_path))
        is_rep = (img_path == res['representative_path'])
        con.execute(
            'INSERT INTO v5_clusters VALUES (?, ?, ?, ?, ?, ?, ?, ?, now())',
            [cid, pid, img_path, is_rep, style, mood, caption, kws]
        )

n_saved = con.execute('SELECT COUNT(*) FROM v5_clusters').fetchone()[0]
n_clust = con.execute('SELECT COUNT(DISTINCT cluster_id) FROM v5_clusters').fetchone()[0]
con.close()
print(f'[✓] {n_saved} photos across {n_clust} clusters saved to v5_clusters')

In [ ]:
# Edit QUERY to search your library by aesthetic description
QUERY = 'dramatic chiaroscuro lighting, deep shadows, street portrait'
TOP_N = 6

print('[i] Searching: ' + QUERY)

bge_tok = AutoTokenizer.from_pretrained(BGE_PATH)
bge_m   = AutoModel.from_pretrained(BGE_PATH, device_map='mps')
bge_m.eval()

# BGE convention: prefix queries (not documents) with this string
query_text = 'Represent this sentence: ' + QUERY
inputs = bge_tok(query_text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
with torch.inference_mode():
    out = bge_m(**inputs)
q_emb = out.last_hidden_state[:, 0, :].detach().cpu().float().numpy().flatten()
q_emb /= (np.linalg.norm(q_emb) + 1e-8)

del bge_m, bge_tok
torch.mps.empty_cache()
gc.collect()

con  = duckdb.connect(str(DB_PATH))
rows = con.execute(
    'SELECT photo_id, image_path, aesthetic_json, text_emb FROM v5_aesthetic WHERE text_emb IS NOT NULL'
).fetchall()
con.close()

scored = []
for pid, img_path, aj, text_emb in rows:
    sim = float(np.dot(q_emb, np.array(text_emb)))
    scored.append((sim, img_path, json.loads(aj)))
scored.sort(reverse=True)
top = scored[:TOP_N]

fig, axes = plt.subplots(1, len(top), figsize=(20, 5))
if len(top) == 1: axes = [axes]
for i, (score, img_path, profile) in enumerate(top):
    cc = get_cc(profile)
    axes[i].imshow(safe_to_rgb(Path(img_path))); axes[i].axis('off')
    label = f'{score:.3f}\n' + cc.get('style', '') + '\n' + cc.get('mood_profile', '')
    axes[i].set_title(label, fontsize=7)

plt.suptitle('V5 Search: ' + QUERY, fontsize=13, y=1.05)
plt.tight_layout(); plt.show()